# Recommendation System
Hybrid: item-based collaborative filtering for users with 2+ ratings (~32% of users), Bayesian-adjusted popularity fallback for cold-start users (~68%, only 1 rating).

In [1]:
import pandas as pd, numpy as np
from sklearn.metrics.pairwise import cosine_similarity

m = pd.read_csv('../data/cleaned/master_clean.csv')
attr = pd.read_csv('../data/cleaned/attractions_clean.csv')
ui = m.pivot_table(index='UserId', columns='AttractionId', values='Rating', aggfunc='mean')
print('User-item matrix:', ui.shape, ' density:', round((ui.notna().sum().sum())/(ui.shape[0]*ui.shape[1])*100,2), '%')

User-item matrix: (33530, 30)  density: 4.5 %


In [2]:
ui_filled = ui.fillna(0)
item_sim = cosine_similarity(ui_filled.T)
item_sim_df = pd.DataFrame(item_sim, index=ui.columns, columns=ui.columns)

agg = m.groupby('AttractionId').agg(avg_rating=('Rating','mean'), n=('Rating','size'))
C = agg.n.mean(); mprior = agg.avg_rating.mean()
agg['bayes_score'] = (agg.n/(agg.n+C))*agg.avg_rating + (C/(agg.n+C))*mprior
popularity_rank = agg.sort_values('bayes_score', ascending=False)

def recommend_for_user(user_id, top_n=5):
    user_ratings = ui.loc[user_id].dropna() if user_id in ui.index else pd.Series(dtype=float)
    if len(user_ratings) == 0:
        return popularity_rank.head(top_n).index.tolist(), 'cold_start_popularity'
    scores = pd.Series(0.0, index=ui.columns); weight_sum = pd.Series(0.0, index=ui.columns)
    for item, rating in user_ratings.items():
        sims = item_sim_df[item]
        scores += sims*rating; weight_sum += sims.abs()
    weight_sum = weight_sum.replace(0, np.nan)
    final = (scores/weight_sum).fillna(0).drop(index=user_ratings.index, errors='ignore')
    return final.sort_values(ascending=False).head(top_n).index.tolist(), 'collaborative_filtering'

recs, method = recommend_for_user(ui.index[0], 5)
attr.set_index('AttractionId').loc[recs, 'Attraction'].tolist(), method

(['Kalibiru National Park',
  'Balekambang Beach',
  'Coban Rondo Waterfall',
  'Mount Semeru Volcano',
  'Sempu Island'],
 'collaborative_filtering')

## Result
The hybrid design is necessary, not optional: a pure collaborative filter would have nothing to recommend for 68% of users. This is the strongest of the three ML components — the underlying co-rating signal is far less noisy than individual rating/visit-mode prediction.